# PersonaPlex + IMTalker — RunPod Live Streaming Notebook

**Pipeline (from source code):**
```
Browser mic (48 kHz PCM, binary WS)
  └─> MoshiOnlyEngineWithHidden  (liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py)
        Mimi encoder  ──> PersonaPlex 7B LM (bnb-4bit)  ──> Mimi decoder
        Layer[-2] hidden states  [B=1, 1, 4096]  @ 12.5 Hz
  └─> StudioNativeLiveAdapter  (helium_w2v_frontend_adapter.py + wav2vec2.py)
        HeliumToWav2VecFrontendAdapter (6-layer Transformer)  ->  [T×768]
        Frozen Wav2Vec2.encode_from_projected_frontend()      ->  [T×768]
  └─> FMGenerator.sample()  (generator/FM.py)
        audio_projection(768→32) + FlowMatchingTransformer + ODE  ->  motion latents [T×32]
  └─> IMTRenderer  (renderer/models.py)
        dense_feature_encoder(ref_img)  ->  identity features
        adapt(motion, g_r)  ->  latent_token_decoder  ->  motion maps
        decode(motion_maps, ref_motion_maps, ref_features)  ->  512×512 RGB @ 25 fps
  └─> ws_av_binary_codec.pack_av_frame()  ->  binary WebSocket -> browser
```

**Checkpoints required:**
| Checkpoint | Path |
|---|---|
| IMTalker generator | `/workspace/IMTalker/checkpoints/generator.ckpt` |
| IMTalker renderer | `/workspace/IMTalker/checkpoints/renderer.ckpt` |
| PersonaPlex bnb-4bit | `/workspace/personaplex_bnb4/model_bnb_4bit.pt` |
| Helium→Wav2Vec2 adapter | `/workspace/exps/personaplex_frontend_adapter/personaplex_helium_w2v_frontend_adapter/checkpoints/phase2_best_wav2vec_final_loss.pt` |
| Wav2Vec2-base-960h | `/workspace/IMTalker/checkpoints/wav2vec2-base-960h/` |
| Mimi tokenizer | from HF `nvidia/personaplex-7b-v1` |
| Voice prompt | `NATM0.pt` from HF `nvidia/personaplex-7b-v1` `voices.tgz` |
| Reference image | `/workspace/IMTalker/assets/2_vid_robert.png` |

**Source files used:**
- `IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py` — main entrypoint
- `IMTalker/liveTry.py` — `MoshiOnlyEngine` base class
- `IMTalker/generator/FM.py` — `FMGenerator`
- `IMTalker/generator/helium_w2v_frontend_adapter.py` — `HeliumToWav2VecFrontendAdapter`
- `IMTalker/generator/wav2vec2.py` — `Wav2VecModel`
- `IMTalker/renderer/models.py` — `IMTRenderer`
- `IMTalker/ws_av_binary_codec.py` — binary frame packing
- `IMTalker/run_personaplex_imtalker_source5_8998.sh` — canonical run arguments
- `personaplex/moshi/moshi/models/loaders.py` — model loaders
- `personaplex/moshi/moshi/models/lm.py` — `LMModel` / `LMGen`

## Cell 1 — System & CUDA Validation

In [ ]:
import subprocess, sys, os, shutil, json, time, pathlib

def _run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    return r.stdout.strip(), r.returncode

print('=== Python ===')
print(sys.version)

print('\n=== CUDA / PyTorch ===')
import torch
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count: {n}')
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU {i}: {props.name}  VRAM={vram_gb:.1f} GB')
        if vram_gb < 16:
            print(f'  WARNING: GPU {i} has {vram_gb:.1f} GB — recommend >=24 GB for bnb-4bit PersonaPlex')
else:
    raise EnvironmentError('No CUDA GPU detected. This pipeline requires CUDA.')

print('\n=== FFmpeg ===')
ffmpeg_path = shutil.which('ffmpeg')
if ffmpeg_path:
    out, _ = _run('ffmpeg -version')
    print(out.splitlines()[0])
else:
    print('ffmpeg not found — installing...')
    os.system('apt-get install -y ffmpeg -qq')

print('\n=== torchaudio ===')
import torchaudio
print(f'torchaudio: {torchaudio.__version__}')

print('\nSystem validation passed.')

## Cell 2 — Repository Path Validation

In [ ]:
# Expected workspace layout (from run_personaplex_imtalker_source5_8998.sh):
#   /workspace/IMTalker                               <- IMTalker repo root
#   /workspace/personaplex_bnb4                       <- PersonaPlex moshi package + weights
#   /workspace/personaplex_bnb4/moshi                <- moshi Python package (in PYTHONPATH)
#   /workspace/exps/personaplex_frontend_adapter/...  <- adapter checkpoint

IMTALKER_ROOT   = '/workspace/IMTalker'
PERSONAPLEX_ROOT = '/workspace/personaplex_bnb4'
MOSHI_PKG       = '/workspace/personaplex_bnb4/moshi'

missing = []
for p in [IMTALKER_ROOT, PERSONAPLEX_ROOT, MOSHI_PKG]:
    exists = pathlib.Path(p).exists()
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {p}')
    if not exists:
        missing.append(p)

if missing:
    raise FileNotFoundError(
        f'Missing required directories: {missing}\n'
        'Clone/copy the repos to /workspace before proceeding.\n'
        '  git clone <imtalker-repo> /workspace/IMTalker\n'
        '  git clone <personaplex-bnb4-repo> /workspace/personaplex_bnb4'
    )

# Check that key source files exist
MAIN_SCRIPT = f'{IMTALKER_ROOT}/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py'
required_sources = [
    MAIN_SCRIPT,
    f'{IMTALKER_ROOT}/liveTry.py',
    f'{IMTALKER_ROOT}/ws_av_binary_codec.py',
    f'{IMTALKER_ROOT}/generator/FM.py',
    f'{IMTALKER_ROOT}/generator/helium_w2v_frontend_adapter.py',
    f'{IMTALKER_ROOT}/generator/wav2vec2.py',
    f'{IMTALKER_ROOT}/generator/options/base_options.py',
    f'{IMTALKER_ROOT}/renderer/models.py',
    f'{IMTALKER_ROOT}/static/index_v3_binary_fullscreen.html',
    f'{MOSHI_PKG}/moshi/models/loaders.py',
    f'{MOSHI_PKG}/moshi/models/lm.py',
]
print()
src_missing = []
for f in required_sources:
    exists = pathlib.Path(f).is_file()
    status = 'OK' if exists else 'MISSING'
    print(f'[{status}] {f}')
    if not exists:
        src_missing.append(f)

if src_missing:
    raise FileNotFoundError(f'Missing source files: {src_missing}')

print('\nAll repository paths validated.')

## Cell 3 — Dependency Installation

Packages derived from `IMTalker/requirement.txt` plus runtime imports traced in the source files.

In [ ]:
# Core packages from IMTalker/requirement.txt
# Plus runtime dependencies traced from source:
#   liveTry.py         -> sentencepiece, huggingface_hub
#   liveTryHelium...py -> fastapi, uvicorn, transformers (Wav2Vec2FeatureExtractor)
#   generator/FM.py    -> torchdiffeq
#   loaders.py         -> safetensors, bitsandbytes (quantize_4bit path)
#   _apply_system_prompts -> sentencepiece

packages = [
    # requirement.txt
    'numpy<2.0.0',
    'pandas<2.2.0',
    'matplotlib<3.9.0',
    'opencv-python<4.10.0',
    'av==12.0.0',
    'flow-vis',
    'albucore==0.0.16',
    'face_alignment==1.4.1',
    'transformers==4.30.2',
    'pytorch-lightning==2.2.1',
    'torchdiffeq==0.2.5',
    'timm==1.0.9',
    'pyyaml',
    'tqdm',
    'librosa',
    'tensorboard',
    'fastapi==0.115.6',
    'pydantic==2.10.4',
    # Runtime deps from source (not in requirement.txt)
    'uvicorn[standard]',
    'websockets',
    'httpx',
    'sentencepiece',
    'huggingface-hub>=0.20.0',
    'safetensors>=0.4.0',
    'bitsandbytes>=0.43.0',   # quantize_4bit path in loaders.py
    'accelerate>=0.26.0',     # used by bitsandbytes quantization
    'pillow',
]

pip_cmd = f'{sys.executable} -m pip install -q ' + ' '.join(f'"{p}"' for p in packages)
print('Installing packages (this may take a few minutes)...')
ret = os.system(pip_cmd)
if ret != 0:
    print('WARNING: pip returned non-zero. Check output above for errors.')
else:
    print('Package installation complete.')

# Install the personaplex moshi package in editable mode so imports resolve
moshi_setup = pathlib.Path(MOSHI_PKG) / 'setup.cfg'
moshi_pyproject = pathlib.Path(MOSHI_PKG) / 'pyproject.toml'
if moshi_setup.exists() or moshi_pyproject.exists():
    ret2 = os.system(f'{sys.executable} -m pip install -q -e {MOSHI_PKG}')
    if ret2 != 0:
        print('WARNING: moshi editable install returned non-zero.')
    else:
        print(f'Installed moshi package from {MOSHI_PKG}')
else:
    # Fallback: add to sys.path so the moshi package is importable directly
    if MOSHI_PKG not in sys.path:
        sys.path.insert(0, MOSHI_PKG)
    print(f'Added {MOSHI_PKG} to sys.path (no setup.cfg found, using path injection)')

## Cell 4 — Python Path Configuration

PYTHONPATH from `run_personaplex_imtalker_source5_8998.sh`:
```bash
export PYTHONPATH=/workspace/IMTalker:/workspace/personaplex_bnb4/moshi
```

In [ ]:
# Inject the same PYTHONPATH that the run script exports.
# Order matters: IMTalker must come first so its local ws_av_binary_codec.py
# and generator/ packages are found before any site-packages.

import sys, os, pathlib

PATHS_TO_ADD = [
    '/workspace/IMTalker',          # main source root (liveTry.py, ws_av_binary_codec, generator/, renderer/)
    '/workspace/personaplex_bnb4/moshi',  # moshi Python package
]

for p in PATHS_TO_ADD:
    if p not in sys.path:
        sys.path.insert(0, p)
    print(f'sys.path += {p}')

# Also set in the subprocess environment so the server process inherits it
existing_pp = os.environ.get('PYTHONPATH', '')
new_pp = ':'.join(PATHS_TO_ADD)
if existing_pp:
    new_pp = new_pp + ':' + existing_pp
os.environ['PYTHONPATH'] = new_pp

# PyTorch CUDA allocator setting from run script
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

print(f'\nPYTHONPATH={os.environ["PYTHONPATH"]}')
print(f'PYTORCH_CUDA_ALLOC_CONF={os.environ["PYTORCH_CUDA_ALLOC_CONF"]}')
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')

# Quick import smoke test for all source modules
print('\nSmoke-testing imports...')
import importlib
for mod in ['ws_av_binary_codec', 'generator.FM', 'generator.wav2vec2',
            'generator.helium_w2v_frontend_adapter', 'renderer.models']:
    try:
        importlib.import_module(mod)
        print(f'  [OK] {mod}')
    except Exception as e:
        print(f'  [FAIL] {mod}: {e}')

print('\nPath configuration complete.')

## Cell 5 — HuggingFace Authentication

In [ ]:
import os

# The run script requires HF_TOKEN to be set:
#   : "${HF_TOKEN:?Set HF_TOKEN in the environment before running this script}"
# Set your token here OR pass it as an environment variable before running this cell.

HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    # Uncomment and fill in your token if not already in the environment:
    # HF_TOKEN = 'hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'
    raise EnvironmentError(
        'HF_TOKEN is not set.\n'
        'Run in your terminal before launching Jupyter:\n'
        '  export HF_TOKEN=hf_XXXX\n'
        'Or set it in this cell: HF_TOKEN = "hf_XXXX"'
    )

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

# Authenticate with huggingface-hub
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print(f'HuggingFace authenticated (token starts with hf_...{HF_TOKEN[-4:]})')

## Cell 6 — Checkpoint Validation & HuggingFace Asset Downloads

Validates all checkpoints identified in `run_personaplex_imtalker_source5_8998.sh`.
Downloads Mimi and voice prompts from `nvidia/personaplex-7b-v1` if missing.

In [ ]:
import pathlib, os
from huggingface_hub import hf_hub_download

# ---------------------------------------------------------------------------
# Canonical paths (from run_personaplex_imtalker_source5_8998.sh)
# ---------------------------------------------------------------------------
GENERATOR_CKPT  = '/workspace/IMTalker/checkpoints/generator.ckpt'
RENDERER_CKPT   = '/workspace/IMTalker/checkpoints/renderer.ckpt'
MOSHI_WEIGHT    = '/workspace/personaplex_bnb4/model_bnb_4bit.pt'
ADAPTER_CKPT    = ('/workspace/exps/personaplex_frontend_adapter'
                   '/personaplex_helium_w2v_frontend_adapter/checkpoints'
                   '/phase2_best_wav2vec_final_loss.pt')
WAV2VEC_DIR     = '/workspace/IMTalker/checkpoints/wav2vec2-base-960h'
REF_IMAGE       = '/workspace/IMTalker/assets/2_vid_robert.png'
VOICE_PROMPT    = 'NATM0.pt'     # filename; fetched from HF voices.tgz
HF_REPO         = 'nvidia/personaplex-7b-v1'

# Loaders.py constants (from personaplex/moshi/moshi/models/loaders.py)
MIMI_NAME            = 'tokenizer-e351c8d8-checkpoint125.safetensors'
TEXT_TOKENIZER_NAME  = 'tokenizer_spm_32k_3.model'

# ---------------------------------------------------------------------------
# Validate local checkpoints (must already exist — not downloadable here)
# ---------------------------------------------------------------------------
LOCAL_REQUIRED = {
    'generator.ckpt':      GENERATOR_CKPT,
    'renderer.ckpt':       RENDERER_CKPT,
    'model_bnb_4bit.pt':   MOSHI_WEIGHT,
    'adapter checkpoint':  ADAPTER_CKPT,
    'wav2vec2-base-960h/': WAV2VEC_DIR,
    'ref image':           REF_IMAGE,
}

all_ok = True
print('=== Local checkpoint validation ===')
for label, path in LOCAL_REQUIRED.items():
    p = pathlib.Path(path)
    exists = p.exists()
    size = ''
    if exists and p.is_file():
        mb = p.stat().st_size / 1024**2
        size = f'  ({mb:.0f} MB)'
    elif exists and p.is_dir():
        n = len(list(p.iterdir()))
        size = f'  ({n} files)'
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {label}: {path}{size}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        'One or more local checkpoints are missing.\n'
        'These must be placed manually — they are not downloadable from public HuggingFace.\n'
        'Contact the model owner or use the file transfer instructions in PERSONAPLEX_IMTALKER_LIVE.md.'
    )

# ---------------------------------------------------------------------------
# Download Mimi tokenizer + text tokenizer from HF if not cached
# (loaders.py: get_mimi() and get_moshi_lm() both need these)
# ---------------------------------------------------------------------------
print(f'\n=== HuggingFace downloads from {HF_REPO} ===')

for hf_name, label in [
    (MIMI_NAME, 'Mimi tokenizer safetensors'),
    (TEXT_TOKENIZER_NAME, 'SentencePiece tokenizer'),
]:
    try:
        local_path = hf_hub_download(HF_REPO, hf_name)
        mb = pathlib.Path(local_path).stat().st_size / 1024**2
        print(f'  [OK] {label}: {local_path}  ({mb:.0f} MB)')
    except Exception as e:
        print(f'  [WARN] {label} download failed: {e}')

# ---------------------------------------------------------------------------
# Download + extract voices.tgz for NATM0.pt voice prompt
# (liveTry.py: _resolve_voice_prompt_path() does this automatically,
#  but we pre-download here for reliability)
# ---------------------------------------------------------------------------
print(f'\n=== Voice prompt: {VOICE_PROMPT} ===')
import tarfile

try:
    voices_tgz = pathlib.Path(hf_hub_download(HF_REPO, 'voices.tgz'))
    voices_dir = voices_tgz.parent / 'voices'
    if not voices_dir.exists():
        print(f'  Extracting {voices_tgz} ...')
        with tarfile.open(voices_tgz, 'r:gz') as tar:
            tar.extractall(path=voices_tgz.parent)
    target = voices_dir / VOICE_PROMPT
    if target.exists():
        print(f'  [OK] {target}  ({target.stat().st_size/1024:.0f} KB)')
        VOICE_PROMPT_PATH = str(target)
    else:
        print(f'  [WARN] {VOICE_PROMPT} not found in extracted voices dir {voices_dir}')
        VOICE_PROMPT_PATH = VOICE_PROMPT  # let liveTry._resolve_voice_prompt_path handle it
except Exception as e:
    print(f'  [WARN] voices.tgz download/extraction failed: {e}')
    VOICE_PROMPT_PATH = VOICE_PROMPT

print('\nCheckpoint validation complete.')

## Cell 7 — Build Argument Namespace

Constructs the `argparse.Namespace` that matches exactly the arguments in
`run_personaplex_imtalker_source5_8998.sh`, using defaults from
`generator/options/base_options.py` and `LiveHeliumFMOptions.initialize()`.

In [ ]:
import argparse, sys

# Simulate parsing the run-script CLI flags as an argparse.Namespace.
# Every value below corresponds 1-to-1 with a flag in the run script or
# a BaseOptions / LiveHeliumFMOptions default.

args = argparse.Namespace(
    # --- Server ---
    host='0.0.0.0',
    port=8998,
    html_path='/workspace/IMTalker/static/index_v3_binary_fullscreen.html',

    # --- IMTalker model paths (from run script) ---
    generator_path='/workspace/IMTalker/checkpoints/generator.ckpt',
    renderer_path='/workspace/IMTalker/checkpoints/renderer.ckpt',
    ref_path='/workspace/IMTalker/assets/2_vid_robert.png',

    # --- Helium→Wav2Vec2 adapter (from run script) ---
    adapter_path=('/workspace/exps/personaplex_frontend_adapter'
                  '/personaplex_helium_w2v_frontend_adapter/checkpoints'
                  '/phase2_best_wav2vec_final_loss.pt'),
    adapter_num_layers=6,        # matches phase2_best checkpoint
    adapter_dropout=0.1,
    stats_path='',               # unused; accepted for compatibility

    # --- Wav2Vec2 (from run script) ---
    wav2vec_model_path='/workspace/IMTalker/checkpoints/wav2vec2-base-960h',
    wav2vec_sec=0.96,            # training window: 0.96s × 25fps = 24 frames

    # --- PersonaPlex / Moshi (from run script) ---
    moshi_root='/workspace/personaplex_bnb4',
    mimi_hf_repo='nvidia/personaplex-7b-v1',
    moshi_weight='/workspace/personaplex_bnb4/model_bnb_4bit.pt',
    mimi_weight='',              # empty → loaded from HF repo
    tokenizer='',                # empty → loaded from HF repo
    quantize_4bit=True,          # PersonaPlex bnb-4bit
    num_codebooks=8,
    moshi_context=0,
    voice_prompt='NATM0.pt',
    voice_prompt_dir='',         # resolved by _resolve_voice_prompt_path()
    text_prompt=(
        'You work for North South University which is a university and your name is '
        'Nabeel Mohammed. Information: you are answering Computer science related '
        'questions explicitly about models and telling about how moshi and personaplex '
        'are trained to ordinary people. So in lighter terms.'
    ),
    moshi_reply_device='cuda',
    enable_moshi_reply=True,     # mic → PersonaPlex reply → Helium → FM → avatar
    direct_reply_hidden=True,    # use Moshi layer[-2] hidden directly (no re-encode)
    moshi_cfg_coef=1.0,

    # --- FM / audio chunk settings (from run script) ---
    audio_chunk_sec=0.96,
    fm_chunk_frames=24,          # 0.96s × 25fps = 24 frames
    reply_hidden_steps_per_chunk=0,  # 0 → derived: round(24 × 12.5 / 25) = 12
    prebuffer_chunks=0,
    frame_q_backpressure=160,
    audio_path='',               # no file-drive mode; live mic only

    # --- Pose (run script passes no static pose flags → data dict has no 'pose') ---
    static_pose_zero=False,
    static_pose_values=None,

    # --- Renderer ---
    render_sub_batch=8,
    jpeg_quality=58,             # from run script

    # --- Flow Matching ODE (from run script) ---
    a_cfg_scale=1.34,
    nfe=5,
    torchdiffeq_ode_method='euler',
    ode_atol=1e-5,
    ode_rtol=1e-5,

    # --- Shared noise (from run script) ---
    shared_noise=True,
    noise_seed=42,
    noise_max_frames=5000,

    # --- Precision (from run script: --fp32 --tf32) ---
    fp32=True,
    tf32=True,
    compile_renderer=False,

    # --- Session management ---
    device='cuda',
    buffer_ms=80,
    dump_motion=True,
    dump_dir='/workspace/IMTalker/live_try_dumps_personaplex_frontend_source5_cfg134',

    # --- BaseOptions defaults used by FMGenerator ---
    fps=25.0,
    sampling_rate=16000,
    input_size=256,
    input_nc=3,
    seed=42,
    fix_noise_seed=False,
    only_last_features=True,
    average_emotion=False,
    audio_marcing=2,
    attention_window=5,
    audio_dropout_prob=0.1,
    ref_dropout_prob=0.1,
    emotion_dropout_prob=0.1,
    style_dim=512,
    dim_a=512,
    dim_h=512,
    dim_e=7,
    dim_motion=32,
    dim_c=32,
    dim_w=32,
    fmt_depth=8,
    num_heads=8,
    mlp_ratio=4.0,
    no_learned_pe=False,
    num_prev_frames=10,
    max_grad_norm=1.0,
    swin_res_threshold=128,
    window_size=8,
    # audio_adapter_mode controls AudioBridge path in FMGenerator
    # 'none' means audio_projection is a linear from audio_input_dim=768 -> dim_c=32
    audio_adapter_mode='none',
    audio_feat_dim=768,
    audio_adapter_dim=512,
    # Used by FM.sample() to distinguish Helium-driven (direct_reply_hidden) path
    rank='cuda',
    pretrained_dir='./checkpoints',
    # file_chunk_lookahead: unused (no audio_path)
    file_chunk_lookahead=0,
)

print('Argument namespace built:')
for k, v in sorted(vars(args).items()):
    print(f'  {k:35s} = {v!r}')

## Cell 8 — In-Process Server (Option A: Integrated)

Builds the FastAPI app directly from `liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.build_app()`
and launches it in a background thread via uvicorn. This is the same entrypoint as the run script —
no code duplication, no reimplementation.

**Skip this cell and use Cell 9 (subprocess) if you prefer isolation.**

In [ ]:
import threading, uvicorn, logging

# Import the actual build_app function from the source file.
# liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary imports:
#   ws_av_binary_codec, generator.FM, generator.helium_w2v_frontend_adapter,
#   generator.wav2vec2, liveTry (MoshiOnlyEngine), renderer.models
# All of these resolve via the PYTHONPATH set in Cell 4.

import importlib.util, pathlib

# Load the entrypoint module by file path (avoids module name collision).
spec = importlib.util.spec_from_file_location(
    'imtalker_live',
    '/workspace/IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py',
)
imtalker_live = importlib.util.module_from_spec(spec)
spec.loader.exec_module(imtalker_live)

print('[notebook] Building FastAPI app...')
app = imtalker_live.build_app(args)
print('[notebook] App built. Starting uvicorn on 0.0.0.0:8998 ...')

class _UvicornThread(threading.Thread):
    """Runs uvicorn in a daemon thread so Jupyter stays responsive."""
    def __init__(self, app, host, port):
        super().__init__(daemon=True, name='uvicorn-server')
        config = uvicorn.Config(
            app,
            host=host,
            port=port,
            log_level='info',
            ws_ping_interval=20,
            ws_ping_timeout=30,
        )
        self.server = uvicorn.Server(config)

    def run(self):
        self.server.run()

    def stop(self):
        self.server.should_exit = True

_server_thread = _UvicornThread(app, args.host, args.port)
_server_thread.start()

# Wait for the server to be ready
import time, httpx
for attempt in range(30):
    time.sleep(2)
    try:
        r = httpx.get(f'http://127.0.0.1:{args.port}/health', timeout=3)
        if r.status_code == 200:
            data = r.json()
            print(f'[notebook] Health check OK: {data}')
            break
    except Exception:
        print(f'[notebook] Waiting for server... attempt {attempt+1}/30')
else:
    raise RuntimeError('Server did not become healthy after 60s')

## Cell 9 — Subprocess Server (Option B: Isolated)

Launches the run script as a managed subprocess exactly as `run_personaplex_imtalker_source5_8998.sh` does.
**Use this instead of Cell 8 if you want full process isolation.**
Only run ONE of Cell 8 or Cell 9.

In [ ]:
# ── OPTION B: subprocess launch ──────────────────────────────────────────────
# Uncomment and run this cell INSTEAD of Cell 8 if you prefer process isolation.
# ─────────────────────────────────────────────────────────────────────────────

# import subprocess, os, sys, time, pathlib, httpx, threading
#
# LOG_PATH = '/workspace/IMTalker/logs/live_personaplex_imtalker_source5_8998.log'
# pathlib.Path(LOG_PATH).parent.mkdir(parents=True, exist_ok=True)
#
# env = os.environ.copy()
# env['PYTHONPATH']           = '/workspace/IMTalker:/workspace/personaplex_bnb4/moshi'
# env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# env['CUDA_VISIBLE_DEVICES'] = '0'
# # HF_TOKEN must already be in env from Cell 5
#
# cmd = [
#     sys.executable, '-u',
#     '/workspace/IMTalker/liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py',
#     '--generator_path', '/workspace/IMTalker/checkpoints/generator.ckpt',
#     '--renderer_path',  '/workspace/IMTalker/checkpoints/renderer.ckpt',
#     '--adapter_path',   ('/workspace/exps/personaplex_frontend_adapter'
#                          '/personaplex_helium_w2v_frontend_adapter/checkpoints'
#                          '/phase2_best_wav2vec_final_loss.pt'),
#     '--adapter_num_layers', '6',
#     '--adapter_dropout',    '0.1',
#     '--wav2vec_model_path', '/workspace/IMTalker/checkpoints/wav2vec2-base-960h',
#     '--ref_path',           '/workspace/IMTalker/assets/2_vid_robert.png',
#     '--host', '0.0.0.0',
#     '--port', '8998',
#     '--device', 'cuda',
#     '--enable_moshi_reply',
#     '--direct_reply_hidden',
#     '--moshi_root',          '/workspace/personaplex_bnb4',
#     '--mimi_hf_repo',        'nvidia/personaplex-7b-v1',
#     '--moshi_weight',        '/workspace/personaplex_bnb4/model_bnb_4bit.pt',
#     '--quantize_4bit',
#     '--num_codebooks', '8',
#     '--moshi_reply_device', 'cuda',
#     '--moshi_cfg_coef', '1.0',
#     '--voice_prompt', 'NATM0.pt',
#     '--text_prompt', (
#         'You work for North South University which is a university and your name is '
#         'Nabeel Mohammed. Information: you are answering Computer science related '
#         'questions explicitly about models and telling about how moshi and personaplex '
#         'are trained to ordinary people. So in lighter terms.'
#     ),
#     '--a_cfg_scale', '1.34',
#     '--nfe', '5',
#     '--wav2vec_sec', '0.96',
#     '--audio_chunk_sec', '0.96',
#     '--fm_chunk_frames', '24',
#     '--reply_hidden_steps_per_chunk', '0',
#     '--prebuffer_chunks', '0',
#     '--frame_q_backpressure', '160',
#     '--render_sub_batch', '8',
#     '--jpeg_quality', '58',
#     '--dump_motion',
#     '--dump_dir', '/workspace/IMTalker/live_try_dumps_personaplex_frontend_source5_cfg134',
#     '--shared_noise',
#     '--noise_seed', '42',
#     '--noise_max_frames', '5000',
#     '--fp32',
#     '--tf32',
# ]
#
# log_file = open(LOG_PATH, 'w', buffering=1)
#
# _proc = subprocess.Popen(
#     cmd,
#     stdout=log_file,
#     stderr=subprocess.STDOUT,
#     env=env,
#     cwd='/workspace/IMTalker',
# )
# print(f'[notebook] Server PID={_proc.pid}, log={LOG_PATH}')
#
# for attempt in range(60):
#     time.sleep(3)
#     if _proc.poll() is not None:
#         raise RuntimeError(f'Server exited early (code={_proc.returncode}). Check {LOG_PATH}')
#     try:
#         r = httpx.get('http://127.0.0.1:8998/health', timeout=3)
#         if r.status_code == 200:
#             print(f'[notebook] Health OK: {r.json()}')
#             break
#     except Exception:
#         print(f'[notebook] Waiting... attempt {attempt+1}/60')
# else:
#     raise RuntimeError('Server did not become healthy after 180s')

print('Cell 9 (subprocess mode) is commented out. Running Cell 8 (in-process) instead.')
print('Uncomment Cell 9 if you prefer subprocess isolation.')

## Cell 10 — Healthy Startup Verification

From `PERSONAPLEX_IMTALKER_LIVE.md`, healthy startup includes:
```
frontend-fp32 loaded
using direct Moshi reply hidden
voice prompt: .../NATM0.pt
installed PersonaPlex graphed hidden capture
serving /workspace/IMTalker/static/index_v3_binary_fullscreen.html
Uvicorn running on http://0.0.0.0:8998
```

In [ ]:
import httpx, time

PORT = 8998

# 1. Health endpoint (FastAPI /health route from liveTryHeliumFrontendDeque...)
print('=== Health check ===')
r = httpx.get(f'http://127.0.0.1:{PORT}/health', timeout=10)
health = r.json()
print(f'  status_code: {r.status_code}')
for k, v in health.items():
    print(f'  {k}: {v}')

assert health.get('ok') is True, f'Health endpoint returned ok=False: {health}'

# 2. Index HTML (served by FileResponse from /workspace/IMTalker/static/index_v3_binary_fullscreen.html)
print('\n=== HTML index ===')
r2 = httpx.get(f'http://127.0.0.1:{PORT}/', timeout=10)
html_size = len(r2.content)
print(f'  status_code: {r2.status_code}')
print(f'  content-type: {r2.headers.get("content-type", "")}')
print(f'  size: {html_size} bytes')
# PERSONAPLEX_IMTALKER_LIVE.md expects: 200 35708
if html_size < 10000:
    print(f'  WARNING: HTML smaller than expected ({html_size} < 10000)')
assert r2.status_code == 200, f'Index returned {r2.status_code}'

print('\nServer is healthy and serving the IMTalker frontend.')

## Cell 11 — Access URL

Displays the public URL for the IMTalker frontend. On RunPod, port 8998 is
exposed via the pod's HTTP service URL.

In [ ]:
import os

PORT = 8998

# RunPod exposes HTTP ports via a predictable URL pattern.
# The pod ID is available in RUNPOD_POD_ID environment variable.
pod_id = os.environ.get('RUNPOD_POD_ID', '')
if pod_id:
    public_url = f'https://{pod_id}-{PORT}.proxy.runpod.net'
    print(f'Public URL (RunPod):  {public_url}')
else:
    print('RUNPOD_POD_ID not set — running locally or on a non-RunPod host.')
    public_url = f'http://localhost:{PORT}'

print(f'Local URL:            http://127.0.0.1:{PORT}/')
print(f'WebSocket endpoint:   ws://127.0.0.1:{PORT}/ws/conversation')
print()
print('Open the URL above in a Chromium-based browser.')
print('Click the microphone button to begin the live conversation.')
print()
print('The pipeline (from liveTryHeliumFrontendDequeStaticPoseFP32FM_ws_binary.py):')
print('  Browser mic (48 kHz) → MoshiOnlyEngineWithHidden → HeliumToWav2VecFrontendAdapter')
print('  → FMGenerator.sample() → IMTRenderer.decode() → binary WS → browser')

## Cell 12 — Real-Time Monitoring

Polls the `/health` endpoint and streams GPU stats. Run for as long as you need to observe the pipeline.
Interrupt the cell to stop monitoring.

In [ ]:
import time, httpx, subprocess, sys

PORT = 8998
POLL_INTERVAL_S = 10

def _gpu_stats():
    """Returns a one-line GPU VRAM/utilization string."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi',
             '--query-gpu=name,memory.used,memory.total,utilization.gpu,temperature.gpu',
             '--format=csv,noheader,nounits'],
            text=True, timeout=5
        ).strip()
        parts = [p.strip() for p in out.split(',')]
        name, mem_used, mem_total, util, temp = parts
        return (f'{name} | VRAM {mem_used}/{mem_total} MiB '
                f'| GPU util {util}% | {temp}°C')
    except Exception as e:
        return f'nvidia-smi error: {e}'

print(f'Monitoring health every {POLL_INTERVAL_S}s. Interrupt to stop.\n')
try:
    iteration = 0
    while True:
        iteration += 1
        ts = time.strftime('%H:%M:%S')
        try:
            r = httpx.get(f'http://127.0.0.1:{PORT}/health', timeout=5)
            h = r.json()
            uptime = h.get('uptime_sec', 0)
            stage  = h.get('stage', '?')
            loaded = h.get('loaded', '?')
            health_str = f'OK  stage={stage} loaded={loaded} uptime={uptime:.0f}s'
        except Exception as e:
            health_str = f'ERROR: {e}'

        gpu_str = _gpu_stats()
        print(f'[{ts}] #{iteration:04d}  health={health_str}')
        print(f'           gpu={gpu_str}')
        time.sleep(POLL_INTERVAL_S)
except KeyboardInterrupt:
    print('\nMonitoring stopped.')

## Cell 13 — Graceful Shutdown

In [ ]:
# --- Option A shutdown (in-process uvicorn thread) ---
try:
    _server_thread.stop()
    _server_thread.join(timeout=10.0)
    print('[notebook] In-process server stopped.')
except NameError:
    pass  # Cell 8 was not run

# --- Option B shutdown (subprocess) ---
try:
    if _proc.poll() is None:
        _proc.terminate()
        _proc.wait(timeout=15)
        print(f'[notebook] Subprocess PID={_proc.pid} terminated.')
    else:
        print(f'[notebook] Subprocess already exited (code={_proc.returncode}).')
except NameError:
    pass  # Cell 9 was not run

# Release GPU memory
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
print('[notebook] GPU memory released.')

## Appendix A — Architecture Summary

### Model loading sequence (derived from source)

```
LiveHeliumFMEngine.__init__(args)
  ├─ _load_fm(args, device)
  │     FMGenerator(args)  [generator/FM.py]
  │       AudioEncoder (Wav2Vec2, frozen)
  │       FlowMatchingTransformer [generator/FMT.py]
  │       audio_projection: Linear(768→32) + LN + SiLU
  │     torch.load(generator.ckpt)
  │     fm.load_state_dict(cleaned, strict=False)
  ├─ _load_renderer(args, device, dtype)
  │     IMTRenderer(args)  [renderer/models.py]
  │       IdentityEncoder (dense_feature_encoder)
  │       MotionEncoder   (latent_token_encoder)
  │       MotionDecoder   (latent_token_decoder)
  │       SynthesisNetwork (frame_decoder)
  │       IdentidyAdaptive (adapt)
  │       CrossAttention × 6 (imt)
  │     torch.load(renderer.ckpt)
  │     renderer.load_state_dict(cleaned, strict=False)
  ├─ StudioNativeLiveAdapter.__init__(wav2vec_model_path, num_layers=6, dropout=0.1)
  │     HeliumToWav2VecFrontendAdapter(num_layers=6)  [generator/helium_w2v_frontend_adapter.py]
  │       input_norm LN(4096) → input_proj Linear(4096→768) → pos_conv Conv1d(768,768,128) → 6×TransformerEncoderLayer → final_norm
  │     Wav2VecModel.from_pretrained(wav2vec_model_path, local_files_only=True)  [generator/wav2vec2.py]
  │       Frozen Wav2Vec2-base-960h
  │     torch.load(adapter_path)  →  adapter.load_state_dict(state, strict=True)
  ├─ Pre-compute ref image features
  │     dense_feature_encoder(ref_tensor) → (f_r, g_r)
  │     latent_token_encoder(ref_tensor) → ref_x
  │     adapt(ref_x, g_r) → latent_token_decoder → m_r
  └─ MoshiOnlyEngineWithHidden.__init__()   (eager-loaded by enable_moshi_reply=True)
        loaders.get_mimi(mimi_weight, device)  [personaplex/moshi/moshi/models/loaders.py]
        loaders.get_moshi_lm(moshi_weight, dtype=bfloat16, quantize_4bit=True)
        LMGen(lm, cfg_coef=1.0, condition_tensors=cond_tensors)
        mimi.streaming_forever(1)
        lm_gen.streaming_forever(1)
        _apply_system_prompts()   → load_voice_prompt_embeddings(NATM0.pt)
        _install_graph_hidden_capture()   → patches lm_gen._step to return layer_hidden
```

### Per-step inference (80ms chunks @ 12.5 Hz)

```
MoshiOnlyEngineWithHidden._step(pcm24)
  mimi.encode(chunk)               → codes [1, 8, 1]
  lm_gen._step(codes[:, :, :1])    → (tokens, transformer_out, layer_hidden)
  layer_hidden[:1, -1:]            → helium_hidden [1, 1, 4096]  @ 12.5 Hz
  mimi.decode(tokens[:, 1:])       → reply_pcm [1920 samples @ 24kHz]

Every 12 hidden steps (≈0.96s = fm_chunk_frames=24 @ 25fps):
  helium_chunk = cat(12 × [1, 4096])           → [12, 4096]
  StudioNativeLiveAdapter.forward_single(helium_deque[100], target_len=200)
    HeliumToWav2VecFrontendAdapter(helium_deque, target_len=200) → frontend50 [1, 200, 768]
    Wav2Vec2.encode_from_projected_frontend(frontend50)          → final50   [1, 200, 768]
    interpolate(final50, size=24)                                → feat_25   [24, 768]
  FMGenerator.sample(data={a_feat: feat_25, ref_x}, a_cfg_scale=1.34, nfe=5)
    odeint(sample_chunk, x0[B,24,32], linspace(0,1,5))
    → motion [1, 24, 32]
  IMTRenderer:
    adapt(motion, g_r) → latent_token_decoder → m_c (motion maps)
    decode(m_c, m_r, f_r) → frames_rgb [24, 512, 512, 3]
  JPEG encode × 24 frames (ThreadPoolExecutor, 4 workers)
  ws_av_binary_codec.pack_av_frame() × 24 → binary WS → browser
```

### Dependency Report

| Package | Version | Source |
|---|---|---|
| numpy | <2.0.0 | requirement.txt |
| opencv-python | <4.10.0 | requirement.txt |
| transformers | ==4.30.2 | requirement.txt |
| torchdiffeq | ==0.2.5 | requirement.txt; `generator/FM.py` `from torchdiffeq import odeint` |
| timm | ==1.0.9 | requirement.txt |
| fastapi | ==0.115.6 | requirement.txt; server entrypoint |
| pydantic | ==2.10.4 | requirement.txt |
| av | ==12.0.0 | requirement.txt |
| face_alignment | ==1.4.1 | requirement.txt |
| uvicorn | [standard] | `liveTry.py` main() |
| sentencepiece | latest | `liveTry.py` SentencePieceProcessor |
| huggingface-hub | >=0.20 | `liveTry.py` hf_hub_download |
| safetensors | >=0.4.0 | `personaplex/moshi/moshi/models/loaders.py` |
| bitsandbytes | >=0.43.0 | `loaders.get_moshi_lm(quantize_4bit=True)` |
| accelerate | >=0.26.0 | bitsandbytes dependency |

### RunPod Requirements

| Requirement | Value |
|---|---|
| GPU VRAM | ≥24 GB (bnb-4bit PersonaPlex 7B) |
| System RAM | ≥32 GB |
| CUDA | ≥12.1 |
| Port | 8998 (HTTP + WebSocket) |
| HF_TOKEN | Required for `nvidia/personaplex-7b-v1` |
| FFmpeg | System package |

### Known Issues / Code Notes

1. **`get_moshi_lm` `quantize_4bit` parameter** — The standard `kyutai/moshi` loaders do not have this flag. The PersonaPlex fork at `/workspace/personaplex_bnb4/moshi` extends loaders to add it. If `PYTHONPATH` does not put `personaplex_bnb4/moshi` first, `quantize_4bit` will fail.

2. **`_install_graph_hidden_capture` dual path** — The code tries the `prepare_step_input` / `process_transformer_output` API (newer PersonaPlex LMGen), then falls back to monkey-patching `forward_text` on the LM model. Both paths return `(tokens, transformer_out, layer_hidden)` tuples.

3. **`audio_adapter_mode`** — `FMGenerator` in the run configuration uses `mode='none'`, meaning `audio_adapter = nn.Identity()` and `audio_projection = Linear(768→32)+LN+SiLU`. The Wav2Vec2 768-dim features are projected directly to `dim_c=32`.

4. **Reference image** — The run script uses `/workspace/IMTalker/assets/2_vid_robert.png` which is not in the local repo. The repo has `source_1.png` through `source_11.png`. If `2_vid_robert.png` is missing, substitute any 512×512 face image (the renderer resizes to 512×512 via `load_ref_image()`).